# Week 09: Stacks and queues - Solutions

## Assignment

For assignment we ask you to implement a stack and a queue using file i/o operations in sequential access files. This gives you practice with file handling and also reinforce the concepts of stacks and queues.


## Abstract base classes

Classes `QueueADT` and `StackADT` are abstract base classes that define the interface for queues and stacks, respectively. These classes serve as a reminder of the methods that need to be implemented when creating concrete classes for stacks and queues.


In [19]:
from abc import ABC, abstractmethod
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple, Type, Union


class QueueADT(ABC):
    @abstractmethod
    def enqueue(self, item: Any) -> None:
        pass

    @abstractmethod
    def dequeue(self) -> Any:
        pass

    @abstractmethod
    def is_empty(self) -> bool:
        pass

    @abstractmethod
    def size(self) -> int:
        pass

In [ ]:
from abc import ABC, abstractmethod
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple, Type, Union


class StackADT(ABC):
    @abstractmethod
    def push(self, item: Any) -> None:
        pass

    @abstractmethod
    def pop(self) -> Any:
        pass

    @abstractmethod
    def peek(self) -> Any:
        pass

    @abstractmethod
    def is_empty(self) -> bool:
        pass

    @abstractmethod
    def size(self) -> int:
        pass

---

## Code for a file-based queue


In [ ]:
class FileQueue(QueueADT):

    _DEFAULT_FILE: str = "queue.txt"
    _TEMP_FILE: str = "temp_queue.txt"

    def __init__(self, filename: str = _DEFAULT_FILE):
        """Object constructor. It searches for the file in the
        same folder. If the file exists, it opens it in read/write
        mode, scans it, counts its lines, and assigns it to the
        items counter for the queue object. If the file does not exist,
        it creates it in write mode, and initializes the
        items counter to 0.
        """
        self.__filename = filename
        self.__items = 0
        try:
            self.__file = open(self.__filename, "r+")
            for _ in self.__file:
                self.__items += 1
        except FileNotFoundError:
            self.__file = open(self.__filename, "w+")

    def __del__(self):
        """Object destructor. It closes the file."""
        self.__file.close()

    def enqueue(self, item: str):
        """It adds an item to the end of the queue. It writes the item
        to the file, and increases the items counter by 1.
        """
        # close file
        self.__file.close()
        # reopen file in append mode
        self.__file = open(self.__filename, "a")
        self.__file.write(item + "\n")
        self.__items += 1

    def dequeue(self) -> str:
        """It removes an item from the front of the queue. It reads the
        first line of the file, stores it in a variable, and then
        rewrites the file without that line. Finally, it decreases the
        items counter by 1, and returns the stored variable.
        """
        if self.__items == 0:
            raise IndexError("Queue is empty")
        # Bring the file cursor to the beginning of the file by
        # reopening the file in read/write mode
        self.__file.close()
        self.__file = open(self.__filename, "r+")
        # Read the first line and store it in a variable to return later
        item = self.__file.readline().rstrip("\n")
        # Read the rest of the lines and store them in the temp file
        with open(self._TEMP_FILE, "w") as temp_file:
            for line in self.__file:
                temp_file.write(line)
        # Replace the contents of the original file with the temp file
        # The original file is reopened in write mode, which also brings
        # the file cursor to the beginning of the file, and erases its contents.
        with open(self.__filename, "w") as original_file:
            with open(self._TEMP_FILE, "r") as temp_file:
                for line in temp_file:
                    original_file.write(line)
        # Decrease the items counter by 1
        self.__items -= 1
        # Return the stored variable
        return item

    def is_empty(self) -> bool:
        """It returns True if the queue is empty, and False otherwise."""
        return self.__items == 0

    def size(self) -> int:
        """It returns the number of items in the queue."""
        return self.__items

    def __len__(self) -> int:
        """It returns the number of items in the queue."""
        return self.__items

    def __bool__(self):
        """It returns True if the queue is not empty, and False otherwise."""
        return self.__items > 0

    def __str__(self):
        """It returns a string representation of the queue."""
        string = ""
        self.__file.close()
        with open(self.__filename, "r") as file:
            for line in file:
                string += line
        return string

--

## Code for a file-based stack


In [ ]:
class FileStack(StackADT):

    _DEFAULT_FILE: str = "stack.txt"
    _TEMP_FILE: str = "temp_stack.txt"

    def __init__(self, filename: str = _DEFAULT_FILE):
        """Object constructor. It searches for the file in the
        same folder. If the file exists, it opens it in read/write
        mode, scans it, counts its lines, and assigns it to the
        items counter for the stack object. If the file does not exist,
        it creates it also in rad/write mode, and initializes the
        items counter to 0.
        """
        self.__filename = filename
        self.__items = 0
        try:
            self.__file = open(self.__filename, "r+")
            for _ in self.__file:
                self.__items += 1
        except FileNotFoundError:
            self.__file = open(self.__filename, "w+")

    def __del__(self):
        """Object destructor. It closes the file."""
        self.__file.close()

    def push(self, item: str):
        """It adds an item to the top of the stack. The top of the
        stack here is defined as the FIRST line of the file The
        method moves the current contents of the stac into a temp file,
        then overwrites the new item to the original file, and
        finally appends the contents of the temp file to the original file.
        It also increases the items counter by 1.
        """
        # close file
        self.__file.close()
        # reopen file in read mode
        self.__file = open(self.__filename, "r")
        # Read the current contents of the stack and store them in the temp file
        with open(self._TEMP_FILE, "w") as temp_file:
            for line in self.__file:
                temp_file.write(line)
        # Replace the contents of the original file with the new item and the temp file
        with open(self.__filename, "w") as original_file:
            original_file.write(item + "\n")
            with open(self._TEMP_FILE, "r") as temp_file:
                for line in temp_file:
                    original_file.write(line)
        # Increase the items counter by 1
        self.__items += 1

    def pop(self) -> str:
        """It removes an item from the top of the stack. The top of the
        stack here is defined as the FIRST line of the file The method
        reads the first line of the file, stores it in a variable, and
        then rewrites the file without that line. Finally, it decreases
        the items counter by 1, and returns the stored variable.
        """
        if self.__items == 0:
            raise IndexError("Stack is empty")
        # Bring the file pointer to the beginning of the file by reopening the file in read/write mode
        self.__file.close()
        self.__file = open(self.__filename, "r+")
        # Read the first line and store it in a variable
        item = self.__file.readline().rstrip("\n")
        # Read the rest of the lines and store them in the temp file
        with open(self.TEMP_FILE, "w") as temp_file:
            for line in self.__file:
                temp_file.write(line)
        # Replace the contents of the original file with the temp file
        with open(self.__filename, "w") as original_file:
            with open(self.TEMP_FILE, "r") as temp_file:
                for line in temp_file:
                    original_file.write(line)
        # Decrease the items counter by 1
        self.__items -= 1
        # Return the stored variable
        return item

    def peek(self) -> str:
        """It returns the item at the top of the stack without removing it.
        The top of the stack here is defined as the FIRST line of the file.
        It reads the first line of the file, stores it in a variable, and
        then returns that variable.
        """
        if self.__items == 0:
            raise IndexError("Stack is empty")
        # Bring the file pointer to the beginning of the file by reopening the file in read mode
        self.__file.close()
        self.__file = open(self.__filename, "r")
        # Read the first line and store it in a variable
        item = self.__file.readline().rstrip("\n")
        # Return the stored variable
        return item

    def is_empty(self) -> bool:
        """It returns True if the queue is empty, and False otherwise."""
        return self.__items == 0

    def size(self) -> int:
        """It returns the number of items in the queue."""
        return self.__items

    def __len__(self) -> int:
        """It returns the number of items in the queue."""
        return self.__items

    def __bool__(self):
        """It returns True if the queue is not empty, and False otherwise."""
        return self.__items > 0

    def __str__(self):
        """It returns a string representation of the queue."""
        string = ""
        self.__file.close()
        with open(self.__filename, "r") as file:
            for line in file:
                string += line
        return string


---

## Technical notes

Both the stack and queue implementations use a simple text file to store the elements. The stack implementation appends new elements to the end of the file, while the queue implementation reads from the beginning of the file and removes elements as they are dequeued. This approach allows us to use file I/O operations to manage the data structure, but it is important to note that this is not an efficient way to implement stacks and queues in practice. The file-based approach is primarily for educational purposes to demonstrate how file I/O can be used to manage data structures.

The implementations above are basic and do not include error handling or optimizations. In a real-world application, you would want to consider edge cases, such as trying to dequeue from an empty queue or pop from an empty stack, and handle those situations appropriately.

Consistent with the assignment requirements, the code does not use lists or other in-memory data structures to manage the stack and queue, relying solely on file operations to maintain the state of the data structure. A temporary file is used to store the elements, and the file is updated as elements are added or removed.

The two classes share some common functionality, such as checking if the file is empty and managing the file path, but they implement their respective behaviors according to the principles of stacks (LIFO) and queues (FIFO). It may be possible to refactor the code to reduce redundancy, but the current implementation is straightforward and serves the purpose of demonstrating the concepts of stacks and queues using file I/O. Such refactoring could involve creating a base class that handles common file operations, and then having the stack and queue classes inherit from that base class to implement their specific behaviors.

### What to look for?

* Constructor `__init__()` should anticipate that the underlying file may not exist when the stack or queue is first created. The constructor should handle this case gracefully, perhaps by creating an empty file if it does not already exist. This way, the rest of the methods can assume that the file exists and can read from it without encountering errors.

* Method `size()` requiring a full scan of the file to count the number of elements. This works but it is fraught with inefficiencies. In a real implementation, you would want to maintain a count of the number of elements as a field of the class, e.g., `self._count` and ujpdate it whenever you add or remove elements. This way, the `size()` method can simply return the value of `self._count` without needing to read through the file. If you prefer to scan the entire file to count the number of elements, you should check that the file exists first. If the file does not exist, it means the stack or queue is empty, and you can return 0 immediately without trying to read from a non-existent file.